# F01 - Python essentials for AI

**Hardware:** CPU-only · **Time:** 2-3 hours · **Prerequisite:** basic computer use

This notebook builds the small Python toolkit used throughout the foundations
track: functions, collections, validation, files, and a reproducible experiment
log. You do not need advanced Python. You do need to type the code, run it, and
explain each assertion.

By the end you will have `outputs/run_log.csv` and `outputs/run_log.json`.

In [1]:
from dataclasses import dataclass, asdict
from datetime import datetime, timezone
from pathlib import Path
import csv
import json
import math
import platform
import statistics
import sys

SEED = 42
OUTPUT_DIR = Path("outputs")
OUTPUT_DIR.mkdir(exist_ok=True)

print("Python:", sys.version.split()[0])
print("Platform:", platform.platform())
assert sys.version_info >= (3, 10), "Use Python 3.10 or newer"

Python: 3.12.10
Platform: Windows-11-10.0.26200-SP0


## 1. Values, types, and the cost of a mistake

Python is dynamically typed: a variable can point to an `int`, then a `str`.
That flexibility is useful for experiments and dangerous for data pipelines.
Print the type before you do arithmetic, and convert deliberately.

In [2]:
correct = 37
total = 50
accuracy = correct / total
print("accuracy:", accuracy, type(accuracy))
assert isinstance(accuracy, float)
assert 0.0 <= accuracy <= 1.0

# Integer division and floating-point representation both surprise beginners.
print("7 // 2 =", 7 // 2, "| 0.1 + 0.2 =", 0.1 + 0.2)
assert math.isclose(0.1 + 0.2, 0.3, rel_tol=1e-9)

accuracy: 0.74 <class 'float'>
7 // 2 = 3 | 0.1 + 0.2 = 0.30000000000000004


## 2. Collections: choose the shape that matches the question

| Collection | Use it for | Example |
| --- | --- | --- |
| `list` | ordered sequence, duplicates allowed | predictions |
| `tuple` | fixed record, should not change | `(accuracy, latency)` |
| `dict` | lookup by key | metrics by model name |
| `set` | membership and uniqueness | unique labels |

A list of dictionaries is the most common shape for small experiment records.

In [3]:
records = [
    {"model": "baseline", "correct": 30, "total": 50, "latency_ms": 2.1},
    {"model": "logistic", "correct": 41, "total": 50, "latency_ms": 3.4},
    {"model": "tree", "correct": 43, "total": 50, "latency_ms": 4.8},
]
labels = [row["model"] for row in records]
print("models:", labels)
print("unique latency values:", sorted({row["latency_ms"] for row in records}))
assert len(labels) == 3
assert "tree" in labels

models: ['baseline', 'logistic', 'tree']
unique latency values: [2.1, 3.4, 4.8]


## 3. Functions and validation

A function should have one job and a clear contract. The functions below accept
plain numbers, return a float, and raise `ValueError` for impossible inputs.
Assertions are checks you expect to pass; exceptions are for inputs you cannot
trust.

In [4]:
def accuracy(correct: int, total: int) -> float:
    if total <= 0:
        raise ValueError("total must be positive")
    if not 0 <= correct <= total:
        raise ValueError("correct must be between 0 and total")
    return correct / total


def mean(values: list[float]) -> float:
    if not values:
        raise ValueError("mean needs at least one value")
    return sum(values) / len(values)


def mae(actual: list[float], predicted: list[float]) -> float:
    if len(actual) != len(predicted):
        raise ValueError("actual and predicted must have the same length")
    return sum(abs(a - p) for a, p in zip(actual, predicted)) / len(actual)


print("accuracy:", round(accuracy(41, 50), 3))
print("mae:", round(mae([10, 20, 30], [12, 18, 33]), 3))
assert accuracy(41, 50) == 0.82
assert mae([10, 20, 30], [12, 18, 33]) == 7 / 3

for bad_call in [(51, 50), (5, 0)]:
    try:
        accuracy(*bad_call)
    except ValueError as error:
        print("rejected", bad_call, "->", error)
    else:
        raise AssertionError("invalid input should have raised")

accuracy: 0.82
mae: 2.333
rejected (51, 50) -> correct must be between 0 and total
rejected (5, 0) -> total must be positive


## 4. Comprehensions and generator expressions

Use a comprehension when you transform or filter a whole sequence. Do not nest
three comprehensions to look clever; a named function is easier to debug.

The expression below converts records into `(model, accuracy)` pairs and keeps
only models above 0.80.

In [5]:
scored = [(row["model"], accuracy(row["correct"], row["total"])) for row in records]
strong = [name for name, value in scored if value > 0.80]
print("scores:", [(name, round(value, 3)) for name, value in scored])
print("above 0.80:", strong)
assert strong == ["logistic", "tree"]

scores: [('baseline', 0.6), ('logistic', 0.82), ('tree', 0.86)]
above 0.80: ['logistic', 'tree']


## 5. A dataclass with validation

A dataclass removes boilerplate while keeping a named record. `frozen=True`
prevents accidental edits, and `__post_init__` enforces invariants. If a metric
is outside its valid range, fail at construction rather than three cells later.

In [6]:
@dataclass(frozen=True)
class Run:
    model: str
    accuracy: float
    latency_ms: float
    params: int

    def __post_init__(self):
        if not self.model.strip():
            raise ValueError("model name cannot be empty")
        if not 0.0 <= self.accuracy <= 1.0:
            raise ValueError("accuracy must be in [0, 1]")
        if self.latency_ms <= 0 or self.params < 0:
            raise ValueError("latency must be positive and params non-negative")


runs = [
    Run("baseline", accuracy(30, 50), 2.1, 0),
    Run("logistic", accuracy(41, 50), 3.4, 13),
    Run("tree", accuracy(43, 50), 4.8, 1_201),
]
for run in runs:
    print(run)
assert runs[-1].accuracy == 0.86

Run(model='baseline', accuracy=0.6, latency_ms=2.1, params=0)
Run(model='logistic', accuracy=0.82, latency_ms=3.4, params=13)
Run(model='tree', accuracy=0.86, latency_ms=4.8, params=1201)


## 6. Parse untrusted values without crashing the whole notebook

CSV files, forms, and APIs often deliver strings. A parser should return a clear
value or a clear error. The helper below accepts a numeric string or `None` and
returns `None` for unusable input while printing a warning.

In [7]:
def parse_float(raw, field: str) -> float | None:
    if raw is None or str(raw).strip() == "":
        return None
    try:
        return float(raw)
    except (TypeError, ValueError):
        print(f"warning: {field}={raw!r} is not numeric; using None")
        return None


assert parse_float("3.5", "latency") == 3.5
assert parse_float("n/a", "latency") is None
assert parse_float("", "latency") is None
print("parser checks passed")

parser checks passed


## 7. Persist evidence as JSON and CSV

A result that exists only in a notebook output is hard to compare and easy to
lose. Write a machine-readable JSON file and a spreadsheet-friendly CSV. Read
each file back and verify the round trip.

In [8]:
payload = {
    "created_utc": datetime.now(timezone.utc).isoformat(timespec="seconds"),
    "seed": SEED,
    "runs": [asdict(run) for run in runs],
}
json_path = OUTPUT_DIR / "run_log.json"
json_path.write_text(json.dumps(payload, indent=2), encoding="utf-8")
loaded = json.loads(json_path.read_text(encoding="utf-8"))
assert loaded["seed"] == SEED
assert len(loaded["runs"]) == len(runs)

csv_path = OUTPUT_DIR / "run_log.csv"
with csv_path.open("w", newline="", encoding="utf-8") as handle:
    writer = csv.DictWriter(handle, fieldnames=list(asdict(runs[0]).keys()))
    writer.writeheader()
    writer.writerows(asdict(run) for run in runs)

with csv_path.open(newline="", encoding="utf-8") as handle:
    rows = list(csv.DictReader(handle))
assert len(rows) == 3
print("wrote", json_path, "and", csv_path)
print("csv header:", list(rows[0].keys()))

wrote outputs\run_log.json and outputs\run_log.csv
csv header: ['model', 'accuracy', 'latency_ms', 'params']


## 8. A tiny experiment summary

The best run is not always the most accurate one: latency, parameter count, and
interpretability matter. The summary function below makes the trade-off visible
instead of hiding it in a leaderboard.

In [9]:
best_accuracy = max(runs, key=lambda run: run.accuracy)
fastest = min(runs, key=lambda run: run.latency_ms)
summary = {
    "best_accuracy": best_accuracy.model,
    "fastest": fastest.model,
    "mean_accuracy": statistics.mean(run.accuracy for run in runs),
    "mean_latency_ms": statistics.mean(run.latency_ms for run in runs),
}
print(json.dumps(summary, indent=2))
assert summary["best_accuracy"] == "tree"
assert summary["fastest"] == "baseline"

{
  "best_accuracy": "tree",
  "fastest": "baseline",
  "mean_accuracy": 0.76,
  "mean_latency_ms": 3.433333333333333
}


## Check your work

- [ ] You can explain the difference between a list, tuple, dict, and set.
- [ ] `accuracy`, `mae`, and `Run` reject invalid inputs.
- [ ] `outputs/run_log.json` and `outputs/run_log.csv` exist and contain three runs.
- [ ] You can state one reason the fastest model is not the best model.

**Next:** F02 - Virtual environments, packages, Jupyter, and Google Colab.